# Crossword batch 1/5 — puzzles 1–2

Upload one DIFFERENT batch notebook to each of five Kaggle GPU sessions. Run cells in order; restart after installation if needed. Both methods run on each assigned puzzle. The ten-puzzle dataset, model revision and source are frozen identically in all five notebooks.

**Limits per method/puzzle: 30,000 input+output tokens, 60 calls, 30 expansions.** The full experiment has 20 method runs; this notebook has four. Download its uniquely named ZIP, or its named result and manifest files, before ending the session. Partial results can also be exported.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'tokenizers==0.22.2', 'accelerate==1.14.0',
    'bitsandbytes==0.50.2', 'huggingface_hub==0.36.2',
    'lm-format-enforcer==0.11.3', 'pandas', 'matplotlib'])
# Restart the kernel after installation, then continue with the next cell.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, sys, zipfile
BATCH_ID=1
PROJECT=Path('/kaggle/working/crosswords10_parallel_v1_batch01of05')
PROJECT.mkdir(parents=True, exist_ok=True)
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='444da528b87e7a3c8fc689ea2d776d992934cff6cf1430bda9f72a91ae468770'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        target=(PROJECT/name).resolve()
        if PROJECT.resolve() not in target.parents:raise ValueError('Unsafe archive path')
        content=archive.read(name)
        if target.exists() and target.read_bytes()!=content:
            raise RuntimeError('Existing source differs; use a fresh project directory.')
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():target.write_bytes(content)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT)


## 1. Offline tests and frozen batch assignment
No dataset download or resampling occurs here.


In [ ]:
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', '.', '-v'], check=True)
for module in ('crossword_batches', 'crossword_bench'):sys.modules.pop(module, None)
import crossword_bench as bench
import crossword_batches as batches
plan=json.loads((PROJECT/'data/batch_plan.json').read_text())
all_puzzles=json.loads((PROJECT/'data/puzzles10.json').read_text())
puzzles=batches.batch_puzzles(all_puzzles, plan, BATCH_ID)
print('Assigned batch:', BATCH_ID, '/ 5')
print('Puzzle IDs:', [p['id'] for p in puzzles])


## 2. Load Qwen on one GPU
Uses `Qwen/Qwen2.5-7B-Instruct` in bitsandbytes 4-bit NF4. A model revision is pinned on first load. JSON Schema constrains both generation and evaluation. Move words must contain exactly five English letters. Scores and rating IDs are also validated locally; at most two retries are allowed and all calls/tokens count. Formatting constraints do not guarantee correct clue answers.

The same backend is used for both policies. Results from NF4 should not be pooled with older Ollama/GGUF experiments as if the backend were identical. Model loading can take several minutes.


In [ ]:
import torch, platform, importlib.metadata
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
assert torch.cuda.is_available(),'Enable GPU in Kaggle notebook settings.'
MODEL_ID=plan['model']['id']
pin=dict(model=MODEL_ID, revision=plan['model']['revision'])
bench.write_json(PROJECT/'model_pin.json', pin)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=pin['revision'],trust_remote_code=False)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=pin['revision'],
    quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,
    attn_implementation='sdpa',trust_remote_code=False).eval()
client=bench.HFClient(model,tokenizer,torch)
runtime=dict(**pin,backend='transformers-nf4-json-schema-v2',gpu=torch.cuda.get_device_name(0),
    cuda=torch.version.cuda,python=platform.python_version(),
    packages={name:importlib.metadata.version(name) for name in ('torch','transformers','accelerate','bitsandbytes','huggingface_hub','lm-format-enforcer')})
bench.write_json(PROJECT/'runtime.json',runtime)
(PROJECT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Loaded:',MODEL_ID, runtime['gpu'],pin['revision'])


## 3. Shared settings
Both methods use 30k tokens, 60 calls, 30 expansions, depth 20, branching 3 and frontier 60. All retries and judging calls count.


In [ ]:
import copy
config=copy.deepcopy(bench.DEFAULT)
config.update(max_tokens=30_000, max_calls=60, max_expansions=30)
bench.write_json(PROJECT/'config.json',config)
print(json.dumps(config,indent=2))


## Preflight: validate generation and evaluation before the full run
This small synthetic check makes real model calls and saves their costs separately under `preflight/`. It checks response format and legal placements, without scoring against any evaluation answers. It must pass before the benchmark run.


In [ ]:
import hashlib
preflight_config={**config,'max_expansions':1}
preflight_clues=['Fruit often baked in pies']*10
preflight_key=bench.digest(dict(config=preflight_config,runtime=runtime,
    code=hashlib.sha256(Path(bench.__file__).read_bytes()).hexdigest()))[:12]
preflight_dir=PROJECT/'preflight'/preflight_key
preflight_session=bench.Session(client,preflight_config,preflight_dir/'calls','format-preflight',42)
preflight_result,preflight_trace=bench.solve(preflight_clues,'heuristic_bestfirst',preflight_session,preflight_config)
bench.write_json(preflight_dir/'result.json',preflight_result)
bench.write_json(preflight_dir/'trace.json',preflight_trace)
PREFLIGHT_PASSED=(preflight_result['calls']>=2 and preflight_result['prediction']!='_'*25
    and preflight_result['stop_reason']!='invalid_model_response')
print(json.dumps(preflight_result,indent=2))
assert PREFLIGHT_PASSED,'Preflight failed: inspect the saved raw calls before running the benchmark.'
print('Preflight passed. Structured generation and evaluation produced a legal placement.')


## 4. Run batch 1/5
Runs both methods on the assigned two puzzles. Completed records resume with identical settings.


In [ ]:
RUN_BENCHMARK=True
SEED=42
if RUN_BENCHMARK:
    assert globals().get('PREFLIGHT_PASSED', False), 'Pass the preflight cell first.'
    output=batches.run_batch(all_puzzles, client, config, runtime, PROJECT, plan, BATCH_ID, seed=SEED)
    print('Saved results:', output)
else:
    print('Set RUN_BENCHMARK=True when ready.')


## 5. Compare scores and costs
Word accuracy is the primary score. Also report full-grid success, letter accuracy, token cost, calls and runtime. Summaries use only paired completed puzzles. The paired bootstrap resamples puzzles (not individual letters); it does not correct for related clues across puzzles or replace a larger evaluation. A better score alone does not establish SOTA.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
for directory in sorted((PROJECT/'results').glob('*')):
    if not (directory/'records').exists():continue
    summary=bench.report(directory)
    print(directory.name)
    table=pd.DataFrame(summary);display(table)
    paired_path=directory/'per_question.csv'
    if not paired_path.exists():continue
    paired=pd.read_csv(paired_path)
    delta=(paired['heuristic_bestfirst_word_accuracy']-paired['tot_dfs_word_accuracy']).to_numpy()
    rng=np.random.default_rng(123)
    means=rng.choice(delta,size=(5000,len(delta)),replace=True).mean(axis=1)
    comparison=dict(paired_puzzles=len(delta),mean_word_accuracy_difference=float(delta.mean()),
        bootstrap_95_percent_interval=np.quantile(means,[.025,.975]).tolist())
    bench.write_json(directory/'paired_comparison.json',comparison)
    print('Heuristic minus DFS:',comparison)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].bar(table.method,table.mean_word_accuracy);axes[0].set_ylim(0,1)
    axes[0].set_title('Word accuracy');axes[0].tick_params(axis='x',rotation=15)
    axes[1].bar(table.method,table.mean_total_tokens)
    axes[1].set_title('Mean input + output tokens');axes[1].tick_params(axis='x',rotation=15)
    fig.tight_layout();fig.savefig(directory/'comparison.png',dpi=160);plt.show()


## 6. Download results or a partial checkpoint
This cell works after a kernel restart: it discovers saved runs for this batch without needing run-cell variables. The ZIP includes source, frozen inputs, manifests, raw calls and saved records.


In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
import json, os, shutil, sys, zipfile
PROJECT=Path('/kaggle/working/crosswords10_parallel_v1_batch01of05')
if str(PROJECT) not in sys.path:sys.path.insert(0, str(PROJECT))
import crossword_batches as batches
saved=sorted((PROJECT/'results').glob('*/batch_manifest.json'))
if not saved:print('No saved benchmark runs yet. Run setup, preflight and the batch cell first.')
previous_dir=Path.cwd()
try:
    os.chdir('/kaggle/working')
    for manifest in saved:
        output=manifest.parent
        batches.export_batch(output)
        export=Path('/kaggle/working')/(output.name+'.zip')
        with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
            for path in PROJECT.rglob('*'):
                if path.is_file() and '__pycache__' not in path.parts and path.name!='run.lock':
                    archive.write(path, PROJECT.name/path.relative_to(PROJECT))
        display(FileLink(export.name))
        for suffix in ('results.jsonl', 'manifest.json', 'summary.csv'):
            source=output/(output.name+'_'+suffix)
            if source.exists():
                destination=Path('/kaggle/working')/source.name
                shutil.copy2(source, destination)
                display(FileLink(destination.name))
finally:
    os.chdir(previous_dir)
